In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e8/train.csv
/kaggle/input/competitions/playground-series-s6e8/test.csv


In [2]:
import os
import sys
import gc
import time
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, auc, log_loss
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import lightgbm as lgb
import xgboost as xgb
import catboost as cb


SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True


seed_everything(SEED)

In [3]:
DATA_DIR = "/kaggle/input/competitions/playground-series-s6e8"
TRAIN_PATH = DATA_DIR + "/train.csv"
TEST_PATH = DATA_DIR + "/test.csv"

In [4]:
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

df_train.head()


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  object 
 11  stress_level             636221 non-null  object 
 12  academic_work_impact     647145 non-null  object 
 13  addicted_label           691369 non-null  int64  
dtypes: f

In [6]:
# Inspect Target Distribution
target_counts = df_train["addicted_label"].value_counts()
target_pct = df_train["addicted_label"].value_counts(normalize=True) * 100
print("\n--- Target Class Distribution ---")
for val in [1, 0]:
    print(f"Class {val}: {target_counts[val]:,d} samples ({target_pct[val]:.2f}%)")


--- Target Class Distribution ---
Class 1: 490,474 samples (70.94%)
Class 0: 200,895 samples (29.06%)


In [7]:
# print("=== MISSING VALUES & DUPLICATES ===")
# print(f"Missing values in Train: {df_train.isnull().sum().sum()}")
# print(f"Missing values in Test:  {df_test.isnull().sum().sum()}")
# print(f"Duplicate rows in Train: {df_train.duplicated().sum()}")
# print(f"Duplicate rows in Test:  {df_test.duplicated().sum()}")

# print("\n=== PERCENTAGE OF MISSING VALUES ===")
# missing_train = (df_train.isnull().mean() * 100).rename("Train Missing %")
# missing_test = (df_test.isnull().mean() * 100).rename("Test Missing %")
# missing_df = pd.concat([missing_train, missing_test], axis=1)
# print(missing_df[missing_df.sum(axis=1) > 0])

# print("\n=== FEATURE UNIQUENESS & CARDINALITY (TRAIN) ===")
# cardinality = pd.DataFrame({
#     'Dtype': df_train.dtypes,
#     'Unique Values': df_train.nunique()
# })
# print(cardinality)

In [8]:
# ID_COL = "id" if "id" in df_train.columns else None
# base_num_cols = df_train.select_dtypes(include=[np.number]).columns.drop([ID_COL], errors='ignore').tolist()


# if len(base_num_cols) > 0:
#     plt.figure(figsize=(14, 6))
#     melted_df = df_train[base_num_cols[:8]].melt() 
#     sns.boxplot(data=melted_df, x='variable', y='value', palette="crest")
#     plt.title("Distribution / Boxplots of Base Numerical Features", fontsize=14, weight='bold')
#     plt.xticks(rotation=45)
#     plt.tight_layout()
#     plt.show()

#     plt.figure(figsize=(10, 8))

#     sns.heatmap(
#         df_train[base_num_cols].corr(), 
#         annot=True,
#         fmt=".2f",
#         cmap="YlGnBu",
#         linewidths=0.5,
#         cbar_kws={'label': 'Correlation Coefficient'}
#     )
#     plt.title("Correlation Heatmap (Base Features)", fontsize=14, weight='bold')
#     plt.tight_layout()
#     plt.show()

In [9]:
# TARGET = "addicted_label"
# from sklearn.feature_selection import mutual_info_classif
# base_num_cols = df_train.select_dtypes(include=[np.number]).columns.drop([ID_COL], errors='ignore').tolist()
# if len(base_num_cols) > 0:
#     mi_num_cols = [c for c in base_num_cols if c != TARGET]
#     X_sample = df_train[mi_num_cols].fillna(0)
#     y_sample = df_train[TARGET].astype('category').cat.codes

#     mi_scores_before = mutual_info_classif(X_sample, y_sample, random_state=SEED)
#     mi_before_df = pd.DataFrame({'Feature': mi_num_cols, 'MI': mi_scores_before}).sort_values('MI', ascending=False)

#     plt.figure(figsize=(10, 4))
#     sns.barplot(data=mi_before_df, x='MI', y='Feature', palette="rocket", hue='Feature', legend=False)
#     plt.title("Mutual Information with Target (BEFORE Feature Engineering)", fontsize=12, weight='bold')
#     plt.show()

#     skewness_before = df_train[mi_num_cols].skew().abs().sort_values(ascending=False)
#     top_6_skewed = skewness_before.index[: min(6, len(skewness_before))].tolist()

#     print("Top Skewed Features (Absolute Skewness):")
#     print(df_train[mi_num_cols].skew()[top_6_skewed])

#     fig, axes = plt.subplots(2, 3, figsize=(15, 8))
#     axes = axes.flatten()
#     for i, col in enumerate(top_6_skewed):
#         sns.histplot(df_train[col], kde=True, ax=axes[i], color='crimson')
#         axes[i].set_title(f"{col} (Skew: {df_train[col].skew():.2f})")
#     plt.suptitle("Top Skewed Features", fontsize=14, weight='bold')
#     plt.tight_layout()
#     plt.show()

In [10]:
# # =============================================================================
# # EXPLORATORY DATA ANALYSIS & VISUALIZATION
# # =============================================================================
# fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# # Target Distribution
# sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[0, 0], palette=["#3498db", "#e74c3c"])
# axes[0, 0].set_title("Target Distribution (addicted_label)", fontsize=13, fontweight="bold")
# axes[0, 0].set_xlabel("Addicted Label (0 = No, 1 = Yes)")
# axes[0, 0].set_ylabel("Count")
# for i, v in enumerate(target_counts.values):
#     axes[0, 0].text(i, v * 0.5, f"{v:,}\n({target_pct[i]:.1f}%)", ha="center", color="white", fontweight="bold")

# # Missing Value Percentages (Train vs Test)
# missing_train = (df_train.drop(columns=["addicted_label"]).isnull().sum() / len(df_train)) * 100
# missing_test = (df_test.isnull().sum() / len(df_test)) * 100
# missing_df = pd.DataFrame({"Train Missing (%)": missing_train, "Test Missing (%)": missing_test}).sort_values(by="Train Missing (%)", ascending=False)
# missing_df.plot(kind="bar", ax=axes[0, 1], colormap="viridis", width=0.8)
# axes[0, 1].set_title("Feature Missingness (Train vs Test)", fontsize=13, fontweight="bold")
# axes[0, 1].set_ylabel("Missing (%)")
# axes[0, 1].tick_params(axis="x", rotation=45)

# # Daily Screen Time vs Target
# sns.kdeplot(data=df_train, x="daily_screen_time_hours", hue="addicted_label", common_norm=False, fill=True, ax=axes[0, 2], palette=["#2ecc71", "#e74c3c"])
# axes[0, 2].set_title("Daily Screen Time by Target", fontsize=13, fontweight="bold")
# axes[0, 2].set_xlabel("Daily Screen Time (Hours)")

# # Notifications vs Target
# sns.boxplot(data=df_train, x="addicted_label", y="notifications_per_day", ax=axes[1, 0], palette=["#3498db", "#e74c3c"])
# axes[1, 0].set_title("Notifications / Day by Target", fontsize=13, fontweight="bold")
# axes[1, 0].set_xlabel("Addicted Label")

# # Stress Level Addiction Rate
# stress_rate = df_train.groupby("stress_level")["addicted_label"].mean().reset_index()
# sns.barplot(data=stress_rate, x="stress_level", y="addicted_label", ax=axes[1, 1], palette="mako")
# axes[1, 1].set_title("Addiction Rate by Stress Level", fontsize=13, fontweight="bold")
# axes[1, 1].set_ylabel("Addiction Probability")
# for i, r in stress_rate.iterrows():
#     axes[1, 1].text(i, r["addicted_label"] - 0.08, f"{r['addicted_label']*100:.1f}%", ha="center", color="white", fontweight="bold")

# # Sleep Hours vs Screen Time Hexbin/Scatter
# sample_plot = df_train.dropna(subset=["daily_screen_time_hours", "sleep_hours"]).sample(n=5000, random_state=42)
# sns.scatterplot(data=sample_plot, x="daily_screen_time_hours", y="sleep_hours", hue="addicted_label", alpha=0.4, ax=axes[1, 2], palette=["#3498db", "#e74c3c"])
# axes[1, 2].set_title("Screen Time vs Sleep Hours", fontsize=13, fontweight="bold")
# axes[1, 2].set_xlabel("Screen Time (Hours)")
# axes[1, 2].set_ylabel("Sleep (Hours)")

# plt.tight_layout()
# plt.show()

In [12]:
# =============================================================================
# FEATURE ENGINEERING ENGINE
# =============================================================================
def engineer_features(df_input, is_train=True):
    """
    Engineers comprehensive predictive features for smartphone addiction detection:
    1. Missingness signals & total null counts
    2. Screen time breakdown & activity ratios (social, gaming, study, entertainment)
    3. Interaction intensity metrics (notifications/open, opens/hour, minutes/open)
    4. Weekend vs weekday screen time divergence & balance metrics
    5. Sleep deficit and health pressure indicators
    6. Categorical ordinal and interaction features
    """
    df = df_input.copy()
    
    
    # 1. Missingness Indicators & Missing Count
    raw_cols = [c for c in df.columns if c not in ["id", "addicted_label"]]
    df["num_missing_features"] = df[raw_cols].isnull().sum(axis=1).astype(np.int8)
    for col in ["daily_screen_time_hours", "social_media_hours", "gaming_hours", 
                "work_study_hours", "sleep_hours", "notifications_per_day", 
                "app_opens_per_day", "weekend_screen_time"]:
        df[f"isna_{col}"] = df[col].isnull().astype(np.int8)
    
    # 2. Activity Composition & Proportions
    eps = 1e-4
    # Entertainment hours
    df["entertainment_hours"] = df["social_media_hours"].fillna(0) + df["gaming_hours"].fillna(0)
    
    # Proportion of daily screen time allocated to social media / gaming / entertainment
    df["social_media_ratio"] = df["social_media_hours"] / (df["daily_screen_time_hours"] + eps)
    df["gaming_ratio"] = df["gaming_hours"] / (df["daily_screen_time_hours"] + eps)
    df["entertainment_ratio"] = df["entertainment_hours"] / (df["daily_screen_time_hours"] + eps)
    df["work_study_ratio"] = df["work_study_hours"] / (df["daily_screen_time_hours"] + eps)
    
    # Work to entertainment ratio (Productivity balance)
    df["work_to_entertainment_ratio"] = df["work_study_hours"] / (df["entertainment_hours"] + eps)
    
    # Unaccounted Screen Time (screen time not explained by social, gaming, or work)
    df["unaccounted_screen_time"] = df["daily_screen_time_hours"] - (
        df["social_media_hours"].fillna(0) + df["gaming_hours"].fillna(0) + df["work_study_hours"].fillna(0)
    )
    
    # 3. Weekend vs Weekday Usage Dynamics
    df["weekend_to_daily_ratio"] = df["weekend_screen_time"] / (df["daily_screen_time_hours"] + eps)
    df["weekend_screen_diff"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    
    # Estimated weekly screen time
    df["weekly_screen_time_est"] = 5.0 * df["daily_screen_time_hours"] + 2.0 * df["weekend_screen_time"]
    
    # 4. App Engagement & Notification Dynamics
    df["notifications_per_screen_hour"] = df["notifications_per_day"] / (df["daily_screen_time_hours"] + eps)
    df["app_opens_per_screen_hour"] = df["app_opens_per_day"] / (df["daily_screen_time_hours"] + eps)
    df["notifications_per_open"] = df["notifications_per_day"] / (df["app_opens_per_day"] + eps)
    df["screen_minutes_per_open"] = (df["daily_screen_time_hours"] * 60.0) / (df["app_opens_per_day"] + eps)
    
    # 5. Sleep & Daily Schedule Balance
    df["sleep_to_screen_ratio"] = df["sleep_hours"] / (df["daily_screen_time_hours"] + eps)
    df["total_accounted_day_hours"] = df["daily_screen_time_hours"].fillna(0) + df["work_study_hours"].fillna(0) + df["sleep_hours"].fillna(0)
    df["free_time_proxy"] = 24.0 - df["total_accounted_day_hours"]
    
    # Extreme behavioral flags
    df["is_sleep_deprived"] = (df["sleep_hours"] < 6.0).astype(np.float32)
    df["is_heavy_screen_user"] = (df["daily_screen_time_hours"] > 8.0).astype(np.float32)
    df["high_screen_low_sleep"] = (df["is_heavy_screen_user"] * df["is_sleep_deprived"]).astype(np.float32)
    
    # 6. Age & Demographic Binning
    df["age_group"] = pd.cut(
        df["age"], 
        bins=[0, 20, 25, 30, 100], 
        labels=[0, 1, 2, 3]
    ).astype(float)
    
    # 7. Categorical Ordinal & Tuple Encodings
    stress_map = {"Low": 0, "Medium": 1, "High": 2}
    impact_map = {"No": 0, "Yes": 1}
    gender_map = {"Male": 0, "Female": 1, "Other": 2}
    
    df["stress_level_num"] = df["stress_level"].map(stress_map)
    df["academic_impact_num"] = df["academic_work_impact"].map(impact_map)
    df["gender_num"] = df["gender"].map(gender_map)
    
    # Combined interaction categories
    df["gender_stress"] = df["gender"].fillna("Missing").astype(str) + "_" + df["stress_level"].fillna("Missing").astype(str)
    df["stress_impact"] = df["stress_level"].fillna("Missing").astype(str) + "_" + df["academic_work_impact"].fillna("Missing").astype(str)
    
    # Stress-weighted Screen Time
    df["stress_weighted_screen_time"] = df["daily_screen_time_hours"] * (df["stress_level_num"].fillna(1) + 1.0)

    # constrained imputation: daily >= social + gaming + work_study, 0 violations in the
    # 421,427 complete rows, so a missing component is bounded rather than unknown
    COMP = ['social_media_hours', 'gaming_hours', 'work_study_hours']
    ds = df['daily_screen_time_hours']
    nmc = df[COMP].isna().sum(axis=1)
    dm = ds.isna()
    ksum = df[COMP].sum(axis=1, min_count=1)
    slack = ds - ksum
    cA = dm & (nmc == 0)
    cB = (~dm) & (nmc == 1)
    cD = (~dm) & (nmc == 0)
    df['ci_daily_lb']         = np.where(cA, ksum, np.nan)
    df['ci_missing_comp_ub']  = np.where(cB, slack, np.nan)
    df['ci_missing_comp_mid'] = np.where(cB, np.clip(slack / 2, 0, None), np.nan)
    df['ci_interval_width']   = np.where(cB, slack, np.nan)
    df['ci_slack_exact']      = np.where(cD, slack, np.nan)
    df['ci_case']             = np.select([cA, cB], [1.0, 2.0], default=0.0)
    df['ci_n_missing']        = (dm.astype(int) + nmc).astype(float)
    
    
    return df

print("Engineering features on train & test sets...")
t0 = time.time()
df_train_fe = engineer_features(df_train, is_train=True)
df_test_fe = engineer_features(df_test, is_train=False)

# Categorical label encoding for interaction string columns
cat_str_cols = ["gender", "stress_level", "academic_work_impact", "gender_stress", "stress_impact"]
for c in cat_str_cols:
    le = LabelEncoder()
    combined = pd.concat([df_train_fe[c].astype(str), df_test_fe[c].astype(str)], axis=0)
    le.fit(combined)
    df_train_fe[c + "_cat"] = le.transform(df_train_fe[c].astype(str))
    df_test_fe[c + "_cat"] = le.transform(df_test_fe[c].astype(str))

# Group aggregation features: Mean & Std screen time by demographic clusters
group_keys = ["gender_num", "stress_level_num"]
agg_stats = df_train_fe.groupby(group_keys)["daily_screen_time_hours"].agg(["mean", "std"]).reset_index()
agg_stats.columns = group_keys + ["group_screen_time_mean", "group_screen_time_std"]

df_train_fe = df_train_fe.merge(agg_stats, on=group_keys, how="left")
df_test_fe = df_test_fe.merge(agg_stats, on=group_keys, how="left")

df_train_fe["screen_time_diff_group_mean"] = df_train_fe["daily_screen_time_hours"] - df_train_fe["group_screen_time_mean"]
df_test_fe["screen_time_diff_group_mean"] = df_test_fe["daily_screen_time_hours"] - df_test_fe["group_screen_time_mean"]

# Drop raw string columns
drop_cols = ["id", "addicted_label"] + cat_str_cols
features = [c for c in df_train_fe.columns if c not in drop_cols]

X = df_train_fe[features]
y = df_train_fe["addicted_label"].values
X_test = df_test_fe[features]


print(f"Total feature count: {len(features)}")
print(f"Feature list: {features}")

Engineering features on train & test sets...
Total feature count: 58
Feature list: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'num_missing_features', 'isna_daily_screen_time_hours', 'isna_social_media_hours', 'isna_gaming_hours', 'isna_work_study_hours', 'isna_sleep_hours', 'isna_notifications_per_day', 'isna_app_opens_per_day', 'isna_weekend_screen_time', 'entertainment_hours', 'social_media_ratio', 'gaming_ratio', 'entertainment_ratio', 'work_study_ratio', 'work_to_entertainment_ratio', 'unaccounted_screen_time', 'weekend_to_daily_ratio', 'weekend_screen_diff', 'weekly_screen_time_est', 'notifications_per_screen_hour', 'app_opens_per_screen_hour', 'notifications_per_open', 'screen_minutes_per_open', 'sleep_to_screen_ratio', 'total_accounted_day_hours', 'free_time_proxy', 'is_sleep_deprived', 'is_heavy_screen_user', 'high_screen_low_sleep', 'age_group',

In [13]:
print(X.head())

    age  daily_screen_time_hours  social_media_hours  gaming_hours  \
0  24.0                      NaN                1.83          1.59   
1  19.0                     5.97                1.08           NaN   
2  18.0                     5.09                 NaN           NaN   
3  21.0                     6.42                1.26          1.42   
4  26.0                    11.20                1.87          2.81   

   work_study_hours  sleep_hours  notifications_per_day  app_opens_per_day  \
0              2.11         7.46                  122.0               38.0   
1              3.03         8.22                   76.0               19.0   
2               NaN         6.25                  134.0               60.0   
3              3.36         8.85                  112.0               94.0   
4              1.95         5.25                    NaN                NaN   

   weekend_screen_time  num_missing_features  ...  ci_case  ci_n_missing  \
0                 8.63            

In [14]:
# =============================================================================
# CROSS-VALIDATION FRAMEWORK SETUP
# =============================================================================
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

# Out-Of-Fold Prediction Dictionaries
oof_predictions = {}
test_predictions = {}
model_cv_scores = {}

print(f"✓ Stratified {N_SPLITS}-Fold Cross Validation configured (Seed: {SEED}).")
for fold, (trn_idx, val_idx) in enumerate(folds):
    print(f"  Fold {fold+1}: Train={len(trn_idx):,d} samples, Val={len(val_idx):,d} samples (Pos rate: {y[val_idx].mean():.4f})")

✓ Stratified 10-Fold Cross Validation configured (Seed: 42).
  Fold 1: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 2: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 3: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 4: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 5: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 6: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 7: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 8: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 9: Train=622,232 samples, Val=69,137 samples (Pos rate: 0.7094)
  Fold 10: Train=622,233 samples, Val=69,136 samples (Pos rate: 0.7094)


In [15]:
best_params = {
    'n_estimators': 10000, 
    'early_stopping_rounds': 50,
    'device': 'cuda',
    'learning_rate': 0.1,
    'grow_policy': 'depthwise', 
    'max_depth': 3, 
    'min_child_weight': 3.9568093256350263, 
    'gamma': 0.3714146258129477, 
    'subsample': 0.9632453424869812, 
    'colsample_bytree': 0.5666051777253827, 
    'reg_alpha': 1.6482647757999222e-08, 
    'reg_lambda': 2.4857776965808364e-07, 
    'scale_pos_weight': 1.7584375894869906, 
    'max_bin': 493
}

In [ ]:
# for fold, (trn_idx, val_idx) in enumerate(folds):

#     X_trn, y_trn = X.iloc[trn_idx], y[trn_idx]
#     X_val, y_val = X.iloc[val_idx], y[val_idx]

#     model = xgb.XGBClassifier(
#         **best_params
#     )

#     model.fit(
#         X_trn,
#         y_trn,
#         eval_set=[(X_val, y_val)]
#     )
    

#     preds = model.predict_proba(X_test)[:, 1]
#     break

In [ ]:
fold_preds = []

seeds = [42]

for fold, (trn_idx, val_idx) in enumerate(folds):
    X_trn, y_trn = X.iloc[trn_idx], y[trn_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]

    for seed in seeds:
        model = xgb.XGBClassifier(
            **best_params,
            random_state=seed
        )

        model.fit(
            X_trn,
            y_trn,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        preds = model.predict_proba(X_test)[:, 1]
        fold_preds.append(preds)

test_preds = np.mean(fold_preds, axis=0)

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [01:16:09] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  self.starting_round = model.num_boosted_rounds()
/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:385: UserWarning: [01:16:09] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  self.starting_round = model.num_boosted_rounds()


In [ ]:
# # test_preds = model.predict_proba(X_test)
# # print(test_preds[:, 1])
# print(test_preds[0])

In [ ]:
# final_test_preds = test_preds[:, 1]
final_test_preds = test_preds
OUTPUT_SUB_PATH = "submission.csv"

# Generate submission DataFrame
sub_df = pd.DataFrame({
    "id": df_test["id"],
    "addicted_label": final_test_preds
})

# Save to CSV
sub_df.to_csv(OUTPUT_SUB_PATH, index=False)
print(f" Submission successfully saved to: {OUTPUT_SUB_PATH}")